In [0]:
import os

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
ALL_DATA_DIR = os.path.join(BASE_DIR, "data", "all_data_hhs")
BRONZE_PARQUET_DIR = os.path.join(BASE_DIR, "bronze_output", "parquet_data_hhs")
os.makedirs(ALL_DATA_DIR, exist_ok=True)
os.makedirs(BRONZE_PARQUET_DIR, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"ALL_DATA_DIR: {ALL_DATA_DIR}")
print(f"BRONZE_PARQUET_DIR: {BRONZE_PARQUET_DIR}")

In [0]:
# Reading parquet & create dataframe. 

from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os
filepath_benefits = os.path.join(BRONZE_PARQUET_DIR, 'benefits')

filepath_rates = os.path.join(BRONZE_PARQUET_DIR, 'rates')

def read_parquet(filepath: str) -> DataFrame:
    data_f = spark.read.parquet(filepath)
    return data_f
    
df_benefits = read_parquet(filepath_benefits)
df_rates = read_parquet(filepath_rates)


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def filter_baseline_rate(df: DataFrame) -> DataFrame:
    return (
        df
        .filter(
            (F.col("Age") == "30") &
            (F.col("Tobacco") == "No Preference")
        )
        .select(
            F.col("PlanId"),
            F.col("IndividualRate").cast("double").alias("IndividualRate")
        )
        .filter(F.col("IndividualRate").isNotNull())
        # .dropDuplicates(["PlanId"])
    )


df_rate_baseline = df_rates.transform(filter_baseline_rate)

display(df_rate_baseline)
print(f"Step 1 — baseline rate rows: {df_rate_baseline.count()}")

In [0]:
TARGET_SERVICE_PRIMARY_CARE = "Primary Care Visit to Treat an Injury or Illness"
TARGET_SERVICE_ROUTINE_DENTAL_ADULT = 'Routine Dental Services (Adult)'
TARGET_SERVICE_BASIC_DENTAL_ADULT = 'Basic Dental Care - Adult'
TARGET_SERVICE_EMERGENCY_TRANSPORT = 'Emergency Transportation/Ambulance'

# Step 2 — service benefit rows: 65704
# Step 2 — service benefit rows: 77353
# Step 2 — service benefit rows: 77353
# Step 2 — service benefit rows: 65704

from pyspark.sql import functions as F, DataFrame

def filter_service_benefits(data_df: DataFrame, targetService: str) -> DataFrame:
    return (
        data_df
        .filter(F.col('BenefitName') == targetService)
        .select(F.col('StandardComponentId').alias('PlanId'), 'BenefitName', 'CopayInnTier1')
        .filter(F.col("CopayInnTier1").isNotNull())
        .dropDuplicates(["PlanId"])
    )

# silver_layer_benefits = filter_service_benefits(df_benefits)
#    
df_benefits_primary_care = df_benefits.transform(filter_service_benefits, TARGET_SERVICE_PRIMARY_CARE)
df_benefits_routine_dental = df_benefits.transform(filter_service_benefits, TARGET_SERVICE_ROUTINE_DENTAL_ADULT)
df_benefits_basic_dental = df_benefits.transform(filter_service_benefits, TARGET_SERVICE_BASIC_DENTAL_ADULT)
df_benefits_emergency_transport = df_benefits.transform(filter_service_benefits, TARGET_SERVICE_EMERGENCY_TRANSPORT)

print(f"Step 2 — service benefit rows: {df_benefits_primary_care.count()}")
print(f"Step 2 — service benefit rows: {df_benefits_routine_dental.count()}")
print(f"Step 2 — service benefit rows: {df_benefits_basic_dental.count()}")
print(f"Step 2 — service benefit rows: {df_benefits_emergency_transport.count()}")


In [0]:
# DOLLAR_REGEX =  r"\$(\d+(?:\.\d+)?)"  # matches $30, $0, $15.50
# DOLLAR_REGEX = r"\d{1,3}(,\d{3})*(\.\d+)?" ##r"(?:\d+(?:\.\d+)?)?" 
DOLLAR_REGEX = r"\$([0-9,]+(?:\.[0-9]{2})?)" 


def parse_copay_amount(df: DataFrame) -> DataFrame:
    copay_lower = F.lower(F.col("CopayInnTier1"))

    is_no_charge   = copay_lower.rlike(r"no charge")
    is_coinsurance = copay_lower.rlike(r"coinsurance|%")
    has_dollar     = F.col("CopayInnTier1").rlike(DOLLAR_REGEX)
    #.rlike(r"\$\d")

    copay_dollar = (
        F.when(is_no_charge,   F.lit(0.0))
         .when(is_coinsurance, F.lit(None).cast("double"))  # exclude percentage-based
         .when(has_dollar,
               F.regexp_extract(F.col("CopayInnTier1"), DOLLAR_REGEX, 1).cast("double"))
         .otherwise(F.lit(None).cast("double"))             # 'Not Applicable', etc.
    )

    return (
        df
        .withColumn("copay_dollar", copay_dollar)
        .filter(F.col("copay_dollar").isNotNull())
        .select('PlanId', 'BenefitName', 'CopayInnTier1', 'copay_dollar')
    )

df_copay_primary_care = df_benefits_primary_care.transform(parse_copay_amount)
df_copay_routine_dental = df_benefits_routine_dental.transform(parse_copay_amount)
df_copay_basic_dental = df_benefits_basic_dental.transform(parse_copay_amount)
df_copay_emergency_transport = df_benefits_emergency_transport.transform(parse_copay_amount)


# display(df_copay_clean)
print(f"Step 3 — clean copay rows (flat dollar only): {df_copay_primary_care.count()}")
print(f"Step 3 — clean copay rows (flat dollar only): {df_copay_routine_dental.count()}")
print(f"Step 3 — clean copay rows (flat dollar only): {df_copay_basic_dental.count()}")
print(f"Step 3 — clean copay rows (flat dollar only): {df_copay_emergency_transport.count()}")

In [0]:
# GOLD LAyer starts here wrt joining the 2 data frames.
def join_rate_and_copay(df_rates: DataFrame, df_copay: DataFrame) -> DataFrame:
    return (
        df_rates
        .join(df_copay, on="PlanId", how="inner")
        # .dropDuplicates(["PlanId", "IndividualRate", "copay_dollar"])
        .select("PlanId", "IndividualRate", "copay_dollar", "CopayInnTier1")
    )

df_analysis_primary_care = join_rate_and_copay(df_rate_baseline, df_copay_primary_care)
df_analysis_routine_dental = join_rate_and_copay(df_rate_baseline, df_copay_routine_dental)
df_analysis_basic_dental = join_rate_and_copay(df_rate_baseline, df_copay_basic_dental)
df_analysis_emergency_transport = join_rate_and_copay(df_rate_baseline, df_copay_emergency_transport)

print(f"Step 4 — joined analysis rows: {df_analysis_primary_care.count()}")
print(f"Step 4 — joined analysis rows: {df_analysis_routine_dental.count()}")
print(f"Step 4 — joined analysis rows: {df_analysis_basic_dental.count()}")
print(f"Step 4 — joined analysis rows: {df_analysis_emergency_transport.count()}")

In [0]:
from pyspark.sql import functions as F

def remove_outliers(df: DataFrame, col: str) -> DataFrame:
    quantiles = df.approxQuantile(col, [0.01, 0.99], 0.01)
    lower, upper = quantiles[0], quantiles[1]
    return df.filter((F.col(col) >= lower) & (F.col(col) <= upper))

df_analysis_no_outliers_primary_care = remove_outliers(df_analysis_primary_care, "IndividualRate")
df_analysis_no_outliers_routine_dental = remove_outliers(df_analysis_routine_dental, "IndividualRate")
df_analysis_no_outliers_basic_dental = remove_outliers(df_analysis_basic_dental, "IndividualRate")
df_analysis_no_outliers_emergency_transport = remove_outliers(df_analysis_emergency_transport, "IndividualRate")    

# Filter data to keep only copay > 0
df_analysis_no_outliers_positive_copay_primary_care = df_analysis_no_outliers_primary_care.filter(F.col("copay_dollar") > 0.0)
                                              
df_analysis_no_outliers_positive_copay_routine_dental = df_analysis_no_outliers_routine_dental.filter(F.col(                                        "copay_dollar") > 0.0)
df_analysis_no_outliers_positive_copay_basic_dental = df_analysis_no_outliers_basic_dental.filter(F.col("copay_dollar") > 0.0)
df_analysis_no_outliers_positive_copay_emergency_transport = df_analysis_no_outliers_emergency_transport.filter(F.col("copay_dollar") > 0.0)

print(f"Step 5 — joined analysis rows: {df_analysis_no_outliers_primary_care.count()}")
print(f"Step 5 — joined analysis rows: {df_analysis_no_outliers_routine_dental.count()}")
print(f"Step 5 — joined analysis rows: {df_analysis_no_outliers_basic_dental.count()}")
print(f"Step 5 — joined analysis rows: {df_analysis_no_outliers_emergency_transport.count()}")


In [0]:
GOLD_PARQUET_DIR = os.path.join(BASE_DIR, "gold", "data")
os.makedirs(GOLD_PARQUET_DIR, exist_ok=True)

write(df_analysis_no_outliers_positive_copay_primary_care, f"{GOLD_PARQUET_DIR}/primary_care")
write(df_analysis_no_outliers_positive_copay_routine_dental, f"{GOLD_PARQUET_DIR}/routine_dental")
write(df_analysis_no_outliers_positive_copay_basic_dental, f"{GOLD_PARQUET_DIR}/basic_dental")
write(df_analysis_no_outliers_positive_copay_emergency_transport, f"{GOLD_PARQUET_DIR}/emergency_transport")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

dfs = {
    "Primary Care": df_analysis_no_outliers_positive_copay_primary_care,
    "Routine Dental": df_analysis_no_outliers_positive_copay_routine_dental,
    "Basic Dental": df_analysis_no_outliers_positive_copay_basic_dental,
    "Emergency Transport": df_analysis_no_outliers_positive_copay_emergency_transport
}

fig, axs = plt.subplots(2, 2, figsize=(18, 12))
axs = axs.flatten()

for i, (service, df) in enumerate(dfs.items()):
    pdf = df.toPandas()
    summary = (
        pdf.groupby("copay_dollar")["IndividualRate"]
        .agg(mean_premium="mean", median_premium="median", plan_count="count")
        .reset_index()
        .sort_values("copay_dollar")
    )
    ax1 = axs[i]
    ax1.plot(summary["copay_dollar"], summary["mean_premium"],
             marker="o", linewidth=2, color="steelblue", label="Mean Premium ($)")
    ax1.plot(summary["copay_dollar"], summary["median_premium"],
             marker="s", linewidth=2, linestyle="--", color="darkorange", label="Median Premium ($)")
    ax1.set_xlabel(f"{service} Copay ($)", fontsize=12)
    ax1.set_ylabel("Individual Monthly Premium ($)", fontsize=12)
    ax1.set_title(f"Copay Tier vs Premium — {service}", fontsize=13)
    ax1.grid(axis="y", linestyle="--", alpha=0.4)

    x = summary["copay_dollar"].values
    y = summary["mean_premium"].values
    if len(x) > 1:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        ax1.plot(x, p(x), color="red", linewidth=2, linestyle=":", label="Trend Line (Mean Premium)")

    ax2 = ax1.twinx()
    ax2.bar(summary["copay_dollar"], summary["plan_count"],
            width=1.5, alpha=0.15, color="grey", label="Plan count")
    ax2.set_ylabel("Number of Plans", fontsize=10, color="grey")
    ax2.tick_params(axis="y", labelcolor="grey")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10)

plt.tight_layout()
plt.savefig("copay_linechart_multi.png", dpi=150)
plt.show()

# Group by copay tier → mean and median premium per tier
# summary = (
#     pdf.groupby("copay_dollar")["IndividualRate"]
#     .agg(mean_premium="mean", median_premium="median", plan_count="count")
#     .reset_index()
#     .sort_values("copay_dollar")
# )

# fig, ax1 = plt.subplots(figsize=(12, 6))

# # Mean premium line
# ax1.plot(summary["copay_dollar"], summary["mean_premium"],
#          marker="o", linewidth=2, color="steelblue", label="Mean Premium ($)")
# ax1.plot(summary["copay_dollar"], summary["median_premium"],
#          marker="s", linewidth=2, linestyle="--", color="darkorange", label="Median Premium ($)")
# # ax1.set_xlabel(f"{TARGET_SERVICE} Copay ($)", fontsize=12)
# ax1.set_ylabel("Individual Monthly Premium ($)", fontsize=12)
# # ax1.set_title(f"Copay Tier vs Premium — {TARGET_SERVICE}\n(Hypothesis: higher copay = lower premium)",
# #               fontsize=13)
# ax1.legend(fontsize=10)
# ax1.grid(axis="y", linestyle="--", alpha=0.4)

# # Add trend line (linear regression) for mean premium
# x = summary["copay_dollar"].values
# y = summary["mean_premium"].values
# if len(x) > 1:
#     z = np.polyfit(x, y, 1)
#     p = np.poly1d(z)
#     ax1.plot(x, p(x), color="red", linewidth=2, linestyle=":", label="Trend Line (Mean Premium)")

# # Secondary axis: plan count per tier (bar chart context)
# ax2 = ax1.twinx()
# ax2.bar(summary["copay_dollar"], summary["plan_count"],
#         width=1.5, alpha=0.15, color="grey", label="Plan count")
# ax2.set_ylabel("Number of Plans", fontsize=10, color="grey")
# ax2.tick_params(axis="y", labelcolor="grey")

# lines1, labels1 = ax1.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10)

# plt.tight_layout()
# plt.savefig("copay_linechart.png", dpi=150)
# plt.show()

In [0]:
display(df_benefits.select('BenefitName').distinct())
df_filtered_primary_care = df_benefits.filter(F.col('BenefitName').contains('Routine Dental Services (Adult)')).select('BenefitName')
display(df_filtered_primary_care)

# Unit Tests

In [0]:
# Install a few helpers we prepared for you
%pip uninstall -y databricks_helpers exercise_ev_databricks_unit_tests

# Install the databricks helpers 
# %pip install git+https://github.com/data-derp/databricks_helpers.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/databricks_helpers.git

# # Install the databricks test cases
# %pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_bronze import test_write_e2e

test_write_e2e(dbutils.fs.ls(f"{BASE_DIR}/bronze_output"), spark, display)